In [0]:
import os
from pathlib import Path
from pyspark.sql import functions as F
try:
    import kagglehub
except ModuleNotFoundError:
    !pip install kagglehub
    import kagglehub

# Funções:
Nessa seção serão setadas funções para auxiliar o estudo:

In [0]:
def get_tables_names_from_schema(table_schema:str)->list[str]:
    """
    This function returns all table names inside a table schema.
    parameters:
        table_schema(str): the schema of the table
    returns:
        table_names(list[str]): a list of table names
    """
    table_names = sorted([
        row.tableName
        for row in spark.sql(f"SHOW TABLES IN mvp.{table_schema}").collect()
    ])
    return table_names


In [0]:
def show_table_full_schema(table_schema:str, table_name:str)->None:
    """
    This function shows the schema of a table in the catalog and schema.
    parameters:
        table_schema(str): the schema of the table
        table_name(str): the name of the table
    """
    print(f"SCHEMA FOR MVP.{table_schema.upper()}.{table_name.upper()}")
    spark.sql(f"""
        SELECT table_catalog, table_schema, table_name, comment
        FROM mvp.information_schema.tables
        WHERE table_schema = '{table_schema}'
        AND table_name = '{table_name}'
            """
    ).show(truncate=False)
    spark.sql(f"""
        SELECT column_name, data_type, comment
        FROM mvp.information_schema.columns
        WHERE table_schema = '{table_schema}'
        AND table_name = '{table_name}'
        """
    ).show(truncate=False)

# Perguntas a serem respondidas:
1. Qual categoria gera mais faturamento?
2. Produtos com mais fotos vendem mais?
3. Produtos entregues com atrazo impactam na avaliação?
4. Quais estados possuem maior volume de venda? 


# Download Dataset from kaggle:
Para esse trabalho, foi escolhido o dataset olist_brazilian_ecommerce do kaggle.<br>
Esse dataset foi escolhido por conter multiplas tabelas relacionadas em um contexto de ecommerce brasileiro. Possibilitando uma análise mais complexa e um processo de pipeline mais robusto. 
- dataset name: Brazilian E-Commerce Public Dataset by Olist
- dataset link: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
- dataset licence: [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

In [0]:
dataset_path = "./dataset"

In [0]:
path = kagglehub.dataset_download(
    "olistbr/brazilian-ecommerce",
    output_dir=dataset_path,    
    )

print("Path to dataset files:", path)

Path to dataset files: ./dataset


In [0]:
print(os.listdir(dataset_path))

['.complete', 'olist_geolocation_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_orders_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv', 'olist_customers_dataset.csv']


In [0]:
# Get all csv files in a list
file_paths = [
    os.path.join(dataset_path, file) for file in os.listdir(dataset_path) 
    if os.path.isfile(os.path.join(dataset_path, file)) 
    and str(file).endswith("csv")
]
file_paths

['./dataset/olist_geolocation_dataset.csv',
 './dataset/olist_order_items_dataset.csv',
 './dataset/olist_order_payments_dataset.csv',
 './dataset/olist_order_reviews_dataset.csv',
 './dataset/olist_orders_dataset.csv',
 './dataset/olist_products_dataset.csv',
 './dataset/olist_sellers_dataset.csv',
 './dataset/product_category_name_translation.csv',
 './dataset/olist_customers_dataset.csv']

# Análise exploratória:
Para entender melhor o dataset, foi realizado uma análise exploratória utilizando o polas. Essa análise teve como principal objetivo: 
- Identificar  e compreender as colunas; 
- Identificar possíveis erros;
- Estimar chaves primarias;
- Marcar possiveis relações/chaves estrangeiras;
- Facilitar a criação de um esquema entidade relacionamento do dataset bruto.

Essa análise esta no arquivo ./exploratory_analysis.ipynb

## Esquema ER do dataset bruto:
Como mencionado acima, foi feito o esquema de entidade-relacionamento do dataset bruto utilizando o drawsql.app:<br>
`obs:` Na camada bronze, todos os tipos das colunas são strings, sendo que a diferenciação da tipagem ira ocorrer somente na camada silver.
![raw dataset ER schema](./img/raw_er_schema.jpg)

# Camada Bronze:
Nessa seção, iremos criar a camada bronze do mvp, realizaremos a ingestão dos datasets brutos e adicionaremos os comentários da tabela e das colunas.<br>
Vale comentar, que por serem os dados brutos, iremos fazer a ingestão de todas as colunas como strings, realizando a tipagem em futuras camadas.<br>
Tanto a criação da camada, quanto a ingestão e comentario das tabelas serão feitas de forma automatica no python, enquanto que a inserção dos comentarios das colunas será feito utilizando a ia do databricks, devido a quantidade de colunas e tabelas.<br>

obs. Para a tabela de review, o modo de leitura do csv teve que ter as opções de multiLinear=True e foi setado o quote e escape. O motivo disso foi porque ao ler o csv somente com o separador, ocorreu um deslocamento nas colunas dessa tabela. Esse deslocamento ocorreu devido à coluna review_comment_message, que possui quebra de linha e outros valores de string.

## Passo a passo:
1. Criação e uso do catalogo mvp e do esquema mvp.bronze;
2. Leitura de todos os datasets em uma lista (contendo um dicionario com nome da lista e lista), iterando pela lista de arquivos;
3. Criação de um dicionario com os nome da lista e os comentários de cada tabela;
4. Ingestão ingestão dos dados e adição de comentarios das tabelas iterando pela lista de datasets (do item 2) correlacionando com o dicionario de comentarios (item 3);
5. Adição dos comentários das colunas;

In [0]:
# Create catalog:
spark.sql("CREATE CATALOG IF NOT EXISTS mvp")
spark.sql("CREATE SCHEMA IF NOT EXISTS mvp.bronze")

DataFrame[]

In [0]:
# use schema and catalog:
spark.sql("USE CATALOG mvp")
spark.sql ("USE SCHEMA bronze")

DataFrame[]

In [0]:
raw_tables = []
for file_path in file_paths:
    file_name = os.path.split(file_path)[-1].removesuffix(".csv")
    absolute_path = str(Path(file_path).absolute())
    # for review_column, we will use multiLine option to read the column correctly
    if "review" not in file_name:
        raw_table = spark.read.option("header", True).option("sep", ",").csv(absolute_path)
    else:
        raw_table = spark.read.option("header", True).option("sep", ",").option("multiLine", True).option("quote", '"').option("escape", '"').csv(absolute_path)
    raw_tables.append({file_name: raw_table})
    print(file_name)
    display(raw_table.limit(10))


olist_geolocation_dataset


geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
01037,-23.54562128115268,-46.63929204800168,sao paulo,SP
01046,-23.546081127035535,-46.64482029837157,sao paulo,SP
01046,-23.54612896641469,-46.64295148361138,sao paulo,SP
01041,-23.5443921648681,-46.63949930627844,sao paulo,SP
01035,-23.541577961711493,-46.64160722329613,sao paulo,SP
01012,-23.547762303364266,-46.63536053788448,são paulo,SP
01047,-23.546273112412678,-46.64122516971552,sao paulo,SP
01013,-23.546923208436723,-46.6342636964915,sao paulo,SP
01029,-23.543769055769133,-46.63427784085132,sao paulo,SP
01011,-23.547639550320632,-46.63603162315495,sao paulo,SP


olist_order_items_dataset


order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23 03:55:27,21.90,12.69
00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19.90,11.85
000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10 12:30:45,810.00,70.75
0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26 18:31:29,145.95,11.65
0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06 14:10:56,53.99,11.40


olist_order_payments_dataset


...

seller_id,seller_zip_code_prefix,seller_city,seller_state
3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
c240c4061717ac1806ae6ee72be3533b,20920,rio de janeiro,RJ
e49c26c3edfa46d227d5121a6b6e4d37,55325,brejao,PE
1b938a7ec6ac5061a66a3766e0e75f90,16304,penapolis,SP
768a86e36ad6aae3d03ee3c6433d61df,01529,sao paulo,SP
ccc4bbb5f32a6ab2b7066a4130f114e3,80310,curitiba,PR


product_category_name_translation


product_category_name,product_category_name_english
beleza_saude,health_beauty
informatica_acessorios,computers_accessories
automotivo,auto
cama_mesa_banho,bed_bath_table
moveis_decoracao,furniture_decor
esporte_lazer,sports_leisure
perfumaria,perfumery
utilidades_domesticas,housewares
telefonia,telephony
relogios_presentes,watches_gifts


olist_customers_dataset


customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC
fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,04534,sao paulo,SP
5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG
5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR
4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG


In [0]:
# Create comments of each table:
comments = {
    "olist_customers_dataset": "Contains information about the customer, including customer identifiers and geolocation information, like ZIP code prefix, state and city.",
    "olist_geolocation_dataset": "Contains information about customer's geolocation data, including ZIP code prefix, latitude and longiture coordnates, state and city.",
    "olist_order_items_dataset": "Contains information about the products included in each order, including product seller and order identifiers, price and shipping information, like limit date and freight value.",
    "olist_order_payments_dataset": "Contains information about the payment methods used for each order, including order and payment identifiers, payment method and payment value.",
    "olist_order_reviews_dataset": "Contains information about the reviews left by customers for each order, including review and order identifiers, score, comment title, comment message, creation date and anser timestamp.",
    "olist_orders_dataset": "Contains information about the orders, including order and customer identifiers, order status, purchase timestamp, order approved timestamp and delivery dates information.",
    "olist_products_dataset": "Contains information about the product, including identifier, category, name lenght, description lenght, quantity of photos, weight in grams and lenght, heigh and width in centimeters.",
    "olist_sellers_dataset": "Contains information about the sellers, including identifier, and geolocation information, like ZIP code prefix,state and city.",
    "product_category_name_translation": "Contains information about the product category name translation, including category name and translated name."
}

In [0]:
# Upload tabels in bronze schema and add a comment to each table:
for t in raw_tables:
    name, df = list(t.items())[0]

    # Remove _dataset from table name    
    table_name = name.removesuffix("_dataset") if name.endswith("_dataset") else name
    
    # Get comment
    comment = comments.get(name)
    
    # Create table
    df.write.format("delta").mode("overwrite").saveAsTable(table_name)

    # Add comment
    # Escape single quotes in comment by remove them
    escaped_comment = comment.replace("'", "") if comment else ""
    spark.sql(f"""
              COMMENT ON TABLE mvp.bronze.{table_name} IS 
              '{escaped_comment}'              
    """)


## Esquema das tabelas:
Devido a quantidade de tabelas e colunas nesse dataset, foi utilizado a ia do databricks para realizar os comentarios individuais das colunas de cada tabela. <br>
Isso foi feito da seguinte forma:
1. No menu de navegação, na esquerda, do databricks, abra a aba de catalogo;
2. Nela, vai ter um menu de navegação a direita do menu do databricks, contendo o nome do catalogo. abra o catalogo mvp;
3. Dentro dele tera o esquema bronze (ou outro esquema que você queira acessar, dependendo da seção do trabalho);
4. Dentro dele terão todas as tabelas criadas dentro desse esquema. apara cada tabela:
    1. Acesse a tabela desejada
    2. Na tela terá um esquema da tabela, na parte superior vai ter um botão na parte superior direita para usar a ia para detalhar essa tabela.
    3. Ao apertar no botão, revise os comentários e faça as alterações necessárias, como no caso das colunas com informação monetária, que foram adicionadas a informação referente á moeda utilizada (BRL).
    4. Após isso salve esses comentários.

`obs:` Esse passo a passo não se limita para a camada bronze, sendo assim, nas futuras camadas, ele não será reescrito.

In [0]:
bronze_table_names = get_tables_names_from_schema(table_schema="bronze")
bronze_table_names

['olist_customers',
 'olist_geolocation',
 'olist_order_items',
 'olist_order_payments',
 'olist_order_reviews',
 'olist_orders',
 'olist_products',
 'olist_sellers',
 'product_category_name_translation']

In [0]:
for table_name in bronze_table_names:
    show_table_full_schema(
        table_name=table_name,
        table_schema="bronze"
    )

SCHEMA FOR MVP.BRONZE.OLIST_CUSTOMERS
+-------------+------------+---------------+------------------------------------------------------------------------------------------------------------------------------------------+
|table_catalog|table_schema|table_name     |comment                                                                                                                                   |
+-------------+------------+---------------+------------------------------------------------------------------------------------------------------------------------------------------+
|mvp          |bronze      |olist_customers|Contains information about the customer, including customer identifiers and geolocation information, like ZIP code prefix, state and city.|
+-------------+------------+---------------+------------------------------------------------------------------------------------------------------------------------------------------+

+------------------------+---------+-----

# Camada Prata
Nessa seção iremos:
1. Criar a camada prata do mvp;
2. Realizar a tipagem das colunas;
3. Renomear as colunas, caso necessário (o dataset olist adiciona o nome da tabela no inicio de cada coluna, sendo assim, removeremos o nome da tabela quando necessário);
3. Corrigir erros encontrados durante a análise exploratória;
4. Realizar a limpeza dos dados quando necessário (exclusão da tabela product_category_name_translation, que não será usada);

Nessa camada será realizado somente a exclusão da tabela com as traduções do category_name, não sendo excluído outras tabelas com informações não usadas na camada gold. Essa decisão foi tomada considerando que este trabalho se propõe a responder a apenas 4 perguntas de negócio. Dessa forma, a exclusão de outras tabelas ou colunas poderia resultar na perda de informações relevantes para a elaboração de novas análises e geração de insights em um contexto empresarial real.

In [0]:
# set catalog and create schema:
spark.sql("USE CATALOG mvp")
spark.sql("CREATE SCHEMA IF NOT EXISTS mvp.silver")
spark.sql ("USE SCHEMA silver")

DataFrame[]

## Customers:
nessa tabela, todos as colunas são strings, sendo assim, não sera necessário realizar nenhuma tipagem ou correção.<br>
Portanto, somente iremos renomear as segunintes colunas:
- customer_zip_code_prefix -> zip_code_prefix; 
- customer_city -> city 
- customer_state -> state

In [0]:
table_name = "olist_customers"
table_name

'olist_customers'

In [0]:
df = spark.read.table(f"mvp.bronze.{table_name}")
df.limit(10).display()

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC
fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,04534,sao paulo,SP
5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG
5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR
4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG


In [0]:
# Rename columns:
df = (
    df
    .withColumnRenamed("customer_zip_code_prefix", "zip_code_prefix")
    .withColumnRenamed("customer_city", "city")
    .withColumnRenamed("customer_state", "state")
)
df.limit(10).display()

customer_id,customer_unique_id,zip_code_prefix,city,state
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC
fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,04534,sao paulo,SP
5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG
5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR
4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG


In [0]:
%sql
--check for errors in city:
SELECT *
FROM mvp.bronze.olist_customers
WHERE (
    try_cast(customer_city AS double) IS NOT NULL
)

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state


In [0]:
# Save table in silver layer:
df.write.format("delta").mode("overwrite").saveAsTable(table_name)

## GEOLOCATION:
Nessa tabela, iremos passar os valores de latitude e longitude para double e renomearemos as seguintes colunas:
- geolocation_zip_code_prefix -> zip_code_prefix
- geolocation_lat -> latitude
- geolocation_long -> longitude
- geolocation_city -> city
- geolocation_state -> state

In [0]:
table_name = "olist_geolocation"

In [0]:
df = spark.read.table(f"mvp.bronze.{table_name}")
df.limit(10).display()

geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
01037,-23.54562128115268,-46.63929204800168,sao paulo,SP
01046,-23.546081127035535,-46.64482029837157,sao paulo,SP
01046,-23.54612896641469,-46.64295148361138,sao paulo,SP
01041,-23.5443921648681,-46.63949930627844,sao paulo,SP
01035,-23.541577961711493,-46.64160722329613,sao paulo,SP
01012,-23.547762303364266,-46.63536053788448,são paulo,SP
01047,-23.546273112412678,-46.64122516971552,sao paulo,SP
01013,-23.546923208436723,-46.6342636964915,sao paulo,SP
01029,-23.543769055769133,-46.63427784085132,sao paulo,SP
01011,-23.547639550320632,-46.63603162315495,sao paulo,SP


In [0]:
%sql
-- Check if all latitude values are numeric:
SELECT *
FROM mvp.bronze.olist_geolocation
WHERE (
    try_cast(geolocation_lat AS double) IS NULL OR
    try_cast(geolocation_lng AS double) IS NULL
)

geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state


In [0]:
# In the above query, we saw that all values of latitude and longitude are numeric
# Converting geolocation_lat and geolocation_lng to double
df = (
    df
    .withColumn("geolocation_lat", F.col("geolocation_lat").cast("double"))
    .withColumn("geolocation_lng", F.col("geolocation_lng").cast("double"))
)
df.limit(10).display()

geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
01037,-23.54562128115268,-46.63929204800168,sao paulo,SP
01046,-23.546081127035535,-46.64482029837157,sao paulo,SP
01046,-23.54612896641469,-46.64295148361138,sao paulo,SP
01041,-23.5443921648681,-46.63949930627844,sao paulo,SP
01035,-23.541577961711493,-46.64160722329613,sao paulo,SP
01012,-23.547762303364266,-46.63536053788448,são paulo,SP
01047,-23.546273112412678,-46.64122516971552,sao paulo,SP
01013,-23.546923208436723,-46.6342636964915,sao paulo,SP
01029,-23.543769055769133,-46.63427784085132,sao paulo,SP
01011,-23.547639550320632,-46.63603162315495,sao paulo,SP


In [0]:
# Rename columns
df = (
    df
    .withColumnRenamed("geolocation_zip_code_prefix", "zip_code_prefix")
    .withColumnRenamed("geolocation_lat", "latitude")
    .withColumnRenamed("geolocation_lng", "longitude")
    .withColumnRenamed("geolocation_city", "city")
    .withColumnRenamed("geolocation_state", "state")
)
df.limit(10).display()

zip_code_prefix,latitude,longitude,city,state
01037,-23.54562128115268,-46.63929204800168,sao paulo,SP
01046,-23.546081127035535,-46.64482029837157,sao paulo,SP
01046,-23.54612896641469,-46.64295148361138,sao paulo,SP
01041,-23.5443921648681,-46.63949930627844,sao paulo,SP
01035,-23.541577961711493,-46.64160722329613,sao paulo,SP
01012,-23.547762303364266,-46.63536053788448,são paulo,SP
01047,-23.546273112412678,-46.64122516971552,sao paulo,SP
01013,-23.546923208436723,-46.6342636964915,sao paulo,SP
01029,-23.543769055769133,-46.63427784085132,sao paulo,SP
01011,-23.547639550320632,-46.63603162315495,sao paulo,SP


In [0]:
df.write.format("delta").mode("overwrite").saveAsTable(table_name)

## Order_items:
Nessa tabela iremos passar shipping_limit_date para timestamp e tanto price quanto freight_value para decimal(10,2).<br>
Após isso, como não há necessidade de renomear nenhuma das colunas, vamos salvá-la na camada silver.

In [0]:
table_name = "olist_order_items"

In [0]:
df = spark.read.table(f"mvp.bronze.{table_name}")
df.limit(10).display()

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23 03:55:27,21.90,12.69
00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19.90,11.85
000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10 12:30:45,810.00,70.75
0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26 18:31:29,145.95,11.65
0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06 14:10:56,53.99,11.40


In [0]:
%sql
-- Check if all price values can be casted as decimal:
SELECT *
FROM mvp.bronze.olist_order_items
WHERE (
    try_cast(price AS decimal(10,2)) IS NULL
    OR try_cast(freight_value AS decimal(10,2)) IS NULL
)

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


In [0]:
%sql
-- Check if shipping limit date can be casted as timestamp:
SELECT *
FROM mvp.bronze.olist_order_items
WHERE (
    try_cast(shipping_limit_date AS timestamp) IS NULL
)

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


In [0]:
# As seen in the queries above, all the prices columns can be converted as double(10,2)
# and the shipping_limit_date can be converted as timestamp.
df = (
    df
    .withColumn("price", F.col("price").cast("decimal(10,2)"))
    .withColumn("freight_value", F.col("freight_value").cast("decimal(10,2)"))
    .withColumn("shipping_limit_date", F.col("shipping_limit_date").cast("timestamp"))
)
df.limit(10).display()

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.90,13.29
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.90,19.93
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.00,17.87
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18.000Z,12.99,12.79
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51.000Z,199.90,18.14
00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23T03:55:27.000Z,21.90,12.69
00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14T12:10:31.000Z,19.90,11.85
000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10T12:30:45.000Z,810.00,70.75
0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26T18:31:29.000Z,145.95,11.65
0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06T14:10:56.000Z,53.99,11.40


In [0]:
# Save table in silver layer:
df.write.format("delta").mode("overwrite").saveAsTable(table_name)

## Order_payments:
Nessa tabela vamos transformar a coluna payment_value para decimal(10, 2) e as colunas payment_sequential e payment_installments para inteiros.<br>
Por fim, como não é necessário renomear nenhuma coluna, vamos salvá-las direto na camada silver.

In [0]:
table_name = "olist_order_payments"

In [0]:
df = spark.read.table(f"mvp.bronze.{table_name}")
df.limit(10).display()

order_id,payment_sequential,payment_type,payment_installments,payment_value
b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
298fcdf1f73eb413e4d26d01b25bc1cd,1,credit_card,2,96.12
771ee386b001f06208a7419e4fc1bbd7,1,credit_card,1,81.16
3d7239c394a212faae122962df514ac7,1,credit_card,3,51.84
1f78449c87a54faf9e96e88ba1491fa9,1,credit_card,6,341.09
0573b5e23cbd798006520e1d5b4c6714,1,boleto,1,51.95


In [0]:
%sql
-- Check if payment_sequential and payment_value can be casted as integer and the payment_value as double(10,2):
SELECT *
FROM mvp.bronze.olist_order_payments
WHERE (
    try_cast(payment_sequential AS int) IS NULL
    OR try_cast(payment_installments AS int) IS NULL
    OR try_cast(payment_value AS decimal(10,2)) IS NULL
)

order_id,payment_sequential,payment_type,payment_installments,payment_value


In [0]:
# As all the target columns can be converted without lost of information.
df = (
    df
    .withColumn("payment_sequential", F.col("payment_sequential").cast("INT"))
    .withColumn("payment_installments", F.col("payment_installments").cast("INT"))
    .withColumn("payment_value", F.col("payment_value").cast("decimal(10,2)"))
)
df.limit(10).display()

order_id,payment_sequential,payment_type,payment_installments,payment_value
b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
298fcdf1f73eb413e4d26d01b25bc1cd,1,credit_card,2,96.12
771ee386b001f06208a7419e4fc1bbd7,1,credit_card,1,81.16
3d7239c394a212faae122962df514ac7,1,credit_card,3,51.84
1f78449c87a54faf9e96e88ba1491fa9,1,credit_card,6,341.09
0573b5e23cbd798006520e1d5b4c6714,1,boleto,1,51.95


In [0]:
# save table in silver layer
df.write.format("delta").mode("overwrite").saveAsTable(table_name)

## Order_review:
Nessa tabela iremos converter as colunas review_creation_date e review_answer_timestamp para timestamp e review_score para int.<br> 
Além disso, iremos renomear as seguintes colunas:
- review_score -> score;
- review_comment_title -> title;
- review_comment_message -> message;
- review_creation_date -> creation_date;
- review_answer_timestamp -> answer_timestamp;

In [0]:
table_name = "olist_order_reviews"

In [0]:
df = spark.read.table(f"mvp.bronze.{table_name}")
df.limit(10).display()

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18 00:00:00,2018-01-18 21:46:59
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10 00:00:00,2018-03-11 03:05:13
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17 00:00:00,2018-02-18 14:36:24
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,null,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01 00:00:00,2018-03-02 10:26:53
15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,null,null,2018-04-13 00:00:00,2018-04-16 00:39:37
07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,null,null,2017-07-16 00:00:00,2017-07-18 19:30:34
7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,null,null,2018-08-14 00:00:00,2018-08-14 21:36:06
a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,null,null,2017-05-17 00:00:00,2017-05-18 12:05:37
8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelho esta impresso como 3desinfector e ao chegar esta com outro nome...atualizar com a marca correta uma vez que é o mesmo aparelho,2018-05-22 00:00:00,2018-05-23 16:45:47


In [0]:
%sql
-- validate if we can convert review_score, review_creation_date and review_answer_timestamp without lose information
SELECT *
FROM mvp.bronze.olist_order_reviews
WHERE (
    try_cast(review_score AS int) IS NULL
)
LIMIT 10

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


In [0]:
%sql
-- validate if we can convert review_creation_date and review_answer_timestamp without lose information
SELECT *
FROM mvp.bronze.olist_order_reviews
WHERE (
    try_cast(review_creation_date AS timestamp) IS NULL
    OR (
        try_cast(review_answer_timestamp AS timestamp) IS NULL
        AND try_cast(review_answer_timestamp AS timestamp) IS NOT NULL
    )
)
LIMIT 10

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


In [0]:
# All target collumn can be converted without lose information:
df = (
    df
    .withColumn("review_score", F.col("review_score").cast("INT"))
.withColumn("review_creation_date", F.col("review_creation_date").cast("timestamp"))
.withColumn("review_answer_timestamp", F.col("review_answer_timestamp").cast("timestamp"))
)
df.limit(10).display()
            

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18T00:00:00.000Z,2018-01-18T21:46:59.000Z
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10T00:00:00.000Z,2018-03-11T03:05:13.000Z
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17T00:00:00.000Z,2018-02-18T14:36:24.000Z
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,null,Recebi bem antes do prazo estipulado.,2017-04-21T00:00:00.000Z,2017-04-21T22:02:06.000Z
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01T00:00:00.000Z,2018-03-02T10:26:53.000Z
15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,null,null,2018-04-13T00:00:00.000Z,2018-04-16T00:39:37.000Z
07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,null,null,2017-07-16T00:00:00.000Z,2017-07-18T19:30:34.000Z
7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,null,null,2018-08-14T00:00:00.000Z,2018-08-14T21:36:06.000Z
a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,null,null,2017-05-17T00:00:00.000Z,2017-05-18T12:05:37.000Z
8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelho esta impresso como 3desinfector e ao chegar esta com outro nome...atualizar com a marca correta uma vez que é o mesmo aparelho,2018-05-22T00:00:00.000Z,2018-05-23T16:45:47.000Z


In [0]:
# Rename columns
df = (
    df
    .withColumnRenamed("review_score", "score")
    .withColumnRenamed("review_comment_title", "title")
    .withColumnRenamed("review_comment_message", "message")
    .withColumnRenamed("review_creation_date", "creation_date")
    .withColumnRenamed("review_answer_timestamp", "answer_timestamp")
)
df.limit(10).display()

review_id,order_id,score,title,message,creation_date,answer_timestamp
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18T00:00:00.000Z,2018-01-18T21:46:59.000Z
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10T00:00:00.000Z,2018-03-11T03:05:13.000Z
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17T00:00:00.000Z,2018-02-18T14:36:24.000Z
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,null,Recebi bem antes do prazo estipulado.,2017-04-21T00:00:00.000Z,2017-04-21T22:02:06.000Z
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01T00:00:00.000Z,2018-03-02T10:26:53.000Z
15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,null,null,2018-04-13T00:00:00.000Z,2018-04-16T00:39:37.000Z
07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,null,null,2017-07-16T00:00:00.000Z,2017-07-18T19:30:34.000Z
7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,null,null,2018-08-14T00:00:00.000Z,2018-08-14T21:36:06.000Z
a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,null,null,2017-05-17T00:00:00.000Z,2017-05-18T12:05:37.000Z
8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelho esta impresso como 3desinfector e ao chegar esta com outro nome...atualizar com a marca correta uma vez que é o mesmo aparelho,2018-05-22T00:00:00.000Z,2018-05-23T16:45:47.000Z


In [0]:
df.write.format("delta").mode("overwrite").saveAsTable(table_name)

## Order
Nessa tabela foi necessário converter as colunas order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date e order_estimated_delivery_date para timestamp. <br>
Além disso, as seguintes colunas foram renomeadas:
- order_status -> status
- order_purchase_timestamp -> purchase_timestamp
- order_delivered_customer_date -> delivered_customer_date
- order_delivered_carrier_date -> delivered_carrier_date
- order_estimated_delivery_date -> estimated_delivered_date

In [0]:
table_name = "olist_orders"

In [0]:
df = spark.read.table(f"mvp.bronze.{table_name}")
df.limit(10).display()

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01 00:00:00
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,null,null,2017-05-09 00:00:00
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07 00:00:00
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06 00:00:00
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23 00:00:00


In [0]:
%sql
-- validate if we can convert the columns whitout lose information:
SELECT *
FROM mvp.bronze.olist_orders
WHERE (
    try_cast(order_purchase_timestamp AS timestamp) IS NULL
    OR (
        try_cast(order_approved_at AS timestamp) IS NULL
        AND order_approved_at IS NOT NULL
        )
    OR (
        try_cast(order_delivered_carrier_date AS timestamp) IS NULL
        AND order_delivered_carrier_date IS NOT NULL
        )
    OR (
        try_cast(order_delivered_customer_date AS timestamp) IS NULL
        AND order_delivered_customer_date IS NOT NULL
        )
    OR (
        try_cast(order_estimated_delivery_date AS timestamp) IS NULL
        AND order_estimated_delivery_date IS NOT NULL
        )
)
LIMIT 10

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [0]:
# cast columns as timestamp:
df = (
    df
    .withColumn("order_purchase_timestamp", F.col("order_purchase_timestamp").cast("timestamp"))
    .withColumn("order_approved_at", F.col("order_approved_at").cast("timestamp"))
    .withColumn("order_delivered_carrier_date", F.col("order_delivered_carrier_date").cast("timestamp"))
    .withColumn("order_delivered_customer_date", F.col("order_delivered_customer_date").cast("timestamp"))
    .withColumn("order_estimated_delivery_date", F.col("order_estimated_delivery_date").cast("timestamp"))
)
df.limit(10).display()
    

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09T00:00:00.000Z
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23T18:29:09.000Z,2017-01-25T02:50:47.000Z,2017-01-26T14:16:31.000Z,2017-02-02T14:08:10.000Z,2017-03-06T00:00:00.000Z
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29T11:55:02.000Z,2017-07-29T12:05:32.000Z,2017-08-10T19:45:24.000Z,2017-08-16T17:14:30.000Z,2017-08-23T00:00:00.000Z


In [0]:
# Rename columns:
df = (
    df
    .withColumnRenamed("order_status", "status")
    .withColumnRenamed("order_purchase_timestamp", "purchase_timestamp")
    .withColumnRenamed("order_delivered_customer_date", "delivered_customer_date")
    .withColumnRenamed("order_delivered_carrier_date", "delivered_carrier_date")
    .withColumnRenamed("order_estimated_delivery_date", "estimated_delivery_date")
)
df.limit(10).display()

order_id,customer_id,status,purchase_timestamp,order_approved_at,delivered_carrier_date,delivered_customer_date,estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09T21:57:05.000Z,2017-07-09T22:10:13.000Z,2017-07-11T14:58:04.000Z,2017-07-26T10:57:55.000Z,2017-08-01T00:00:00.000Z
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11T12:22:08.000Z,2017-04-13T13:25:17.000Z,null,null,2017-05-09T00:00:00.000Z
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16T13:10:30.000Z,2017-05-16T13:22:11.000Z,2017-05-22T10:07:46.000Z,2017-05-26T12:55:51.000Z,2017-06-07T00:00:00.000Z
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23T18:29:09.000Z,2017-01-25T02:50:47.000Z,2017-01-26T14:16:31.000Z,2017-02-02T14:08:10.000Z,2017-03-06T00:00:00.000Z
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29T11:55:02.000Z,2017-07-29T12:05:32.000Z,2017-08-10T19:45:24.000Z,2017-08-16T17:14:30.000Z,2017-08-23T00:00:00.000Z


In [0]:
# Save table in silver layer:
df.write.format("delta").mode("overwrite").saveAsTable(table_name)

## Products
Nessa tabela iremos transformar as colunas product_name_lenght, product_description_lenght e product_photos_qty para int e product_weight_g, product_lenght_cm, product_height_cm, product_widht_cm para decimal(8,2).<br>
Além disso, também serão renomeadas as seguintes colunas:
- product_category_name -> category_name;
- product_name_lenght -> name_lenght
- product_description_length -> description_length
- product_photos_qty -> photos_quantity
- product_weight_g -> weight_g
- product_length_cm -> length_cm
- product_height_cm -> height_cm
- product_width_cm -> width_cm

In [0]:
table_name = "olist_products"

In [0]:
df = spark.read.table(f"mvp.bronze.{table_name}")
df.limit(10).display()

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13
41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,60,745,1,200,38,5,11
732bd381ad09e530fe0a5f457d81becb,cool_stuff,56,1272,4,18350,70,24,44
2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,56,184,2,900,40,8,40
37cc742be07708b53a98702e77a21a02,eletrodomesticos,57,163,1,400,27,13,17
8c92109888e8cdf9d66dc7e463025574,brinquedos,36,1156,1,600,17,10,12


In [0]:
%sql
-- validate convertions:
SELECT * 
FROM mvp.bronze.olist_products
WHERE (
    (
        try_cast(product_name_lenght AS int) IS NULL
        AND product_name_lenght IS NOT NULL
    )
    OR ( 
        try_cast(product_description_lenght AS int) IS NULL
        AND product_description_lenght IS NOT NULL
        )
    OR (
        try_cast(product_photos_qty AS int) IS NULL
        AND product_photos_qty IS NOT NULL
        )
)

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm


In [0]:
%sql
-- validate convertions:
SELECT * 
FROM mvp.bronze.olist_products
WHERE (
    try_cast(product_weight_g AS decimal(8, 2)) IS NULL
    OR try_cast(product_length_cm AS decimal(8, 2)) IS NULL
    OR try_cast(product_height_cm AS decimal(8, 2)) IS NULL
    OR try_cast(product_width_cm AS decimal(8, 2)) IS NULL
)

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
09ff539a621711667c43eba6a3bd8466,bebes,60,865,3,null,null,null,null
5eb564652db742ff8f28759cd8d2652a,null,null,null,null,null,null,null,null


In [0]:
# change type of the columns:
df = (
    df
    .withColumn("product_weight_g", F.col("product_weight_g").cast("decimal(8,2)"))
    .withColumn("product_length_cm", F.col("product_length_cm").cast("decimal(8,2)"))
    .withColumn("product_height_cm", F.col("product_height_cm").cast("decimal(8,2)"))
    .withColumn("product_width_cm", F.col("product_width_cm").cast("decimal(8,2)"))
    .withColumn("product_name_lenght", F.col("product_name_lenght").cast("int"))
    .withColumn("product_description_lenght", F.col("product_description_lenght").cast("int"))
    .withColumn("product_photos_qty", F.col("product_photos_qty").cast("int"))
    )
df.limit(10).display()

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225.00,16.00,10.00,14.00
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000.00,30.00,18.00,20.00
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154.00,18.00,9.00,15.00
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371.00,26.00,4.00,26.00
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625.00,20.00,17.00,13.00
41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,60,745,1,200.00,38.00,5.00,11.00
732bd381ad09e530fe0a5f457d81becb,cool_stuff,56,1272,4,18350.00,70.00,24.00,44.00
2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,56,184,2,900.00,40.00,8.00,40.00
37cc742be07708b53a98702e77a21a02,eletrodomesticos,57,163,1,400.00,27.00,13.00,17.00
8c92109888e8cdf9d66dc7e463025574,brinquedos,36,1156,1,600.00,17.00,10.00,12.00


In [0]:
# Rename Columns:
df = (
    df
    .withColumnRenamed("product_category_name", "category_name")
    .withColumnRenamed("product_name_lenght", "name_lenght")
    .withColumnRenamed("product_description_lenght", "description_lenght")
    .withColumnRenamed("product_photos_qty", "photos_quantity")
    .withColumnRenamed("product_weight_g", "weight_g")
    .withColumnRenamed("product_length_cm", "lengt_cm")
    .withColumnRenamed("product_height_cm", "height_cm")
    .withColumnRenamed("product_width_cm", "width_cm")
)


df.limit(10).display()

product_id,category_name,name_lenght,description_lenght,photos_quantity,weight_g,lengt_cm,height_cm,width_cm
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225.00,16.00,10.00,14.00
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000.00,30.00,18.00,20.00
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154.00,18.00,9.00,15.00
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371.00,26.00,4.00,26.00
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625.00,20.00,17.00,13.00
41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,60,745,1,200.00,38.00,5.00,11.00
732bd381ad09e530fe0a5f457d81becb,cool_stuff,56,1272,4,18350.00,70.00,24.00,44.00
2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,56,184,2,900.00,40.00,8.00,40.00
37cc742be07708b53a98702e77a21a02,eletrodomesticos,57,163,1,400.00,27.00,13.00,17.00
8c92109888e8cdf9d66dc7e463025574,brinquedos,36,1156,1,600.00,17.00,10.00,12.00


In [0]:
# Save table in silver layer
df = df.write.format("delta").mode("overwrite").saveAsTable(table_name)


## Sellers
Nessa tabela não faremos nenhuma transformação de tipo, mas iremos renomear as seguintes colunas:
- seller_zip_code_prefix -> zip_code_prefix;
- seller_city -> city;
- seller_state -> state;
Além disso, iremos corrigir um erro encontrado na tabela durante a análise exploratória.

In [0]:
table_name = "olist_sellers"

In [0]:
df = spark.read.table(f"mvp.bronze.{table_name}")
df.limit(10).display()

seller_id,seller_zip_code_prefix,seller_city,seller_state
3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
c240c4061717ac1806ae6ee72be3533b,20920,rio de janeiro,RJ
e49c26c3edfa46d227d5121a6b6e4d37,55325,brejao,PE
1b938a7ec6ac5061a66a3766e0e75f90,16304,penapolis,SP
768a86e36ad6aae3d03ee3c6433d61df,01529,sao paulo,SP
ccc4bbb5f32a6ab2b7066a4130f114e3,80310,curitiba,PR


### Correção de erro:
Durante a análise exploratória, foi identificado um erro dentro da tabela olist_sellers_dataset, mais especificamente na coluna sellers_city. Esse erro consiste no valor "04482255" no campo da cidade.<br>
Para corrigir esse erro, vamos utilizar a tabela de geolocalização, fazendo uma correlação de olist_sellers.sellers_zip_code com olistgeolocation.geolocation_zip_code. Com isso podemos pegar o valor correto da cidade da tabela de geolocalização e atualizar o valor incorreto da tabela sellers.

#### Passo a passo:
1. Mostrar o erro
2. Analisar a tabela de geolocalização para determinar que só existe um valor de cidade para o zip_code desejado
3. Caso exista somente uma cidade, subistituir esse valor pelo valor unico

In [0]:
%sql
-- Show error
SELECT * 
FROM mvp.bronze.olist_sellers
WHERE try_cast(seller_city AS int) IS NOT NULL

seller_id,seller_zip_code_prefix,seller_city,seller_state
ceb7b4fb9401cd378de7886317ad1b47,22790,04482255,RJ


In [0]:
%sql
-- Verify if there is an unique value of seller city with the error
SELECT DISTINCT geolocation_zip_code_prefix, geolocation_city, geolocation_state
FROM mvp.bronze.olist_geolocation 
WHERE geolocation_zip_code_prefix = (
    SELECT seller_zip_code_prefix 
    FROM mvp.bronze.olist_sellers
    WHERE try_cast(seller_city AS int) IS NOT NULL
)

geolocation_zip_code_prefix,geolocation_city,geolocation_state
22790,rio de janeiro,RJ


Como podemos ver acima, existe somente um valor de cidade para esse zip code. Com isso, vamos atualizar a tabela sellers para que seu valor seja 'rio de janeiro'

In [0]:
# Get unique correction for sellers table using geolocation table
geolocation_df = spark.read.table("mvp.silver.olist_geolocation")
sellers_city_correction = (
    geolocation_df
    .filter(geolocation_df.zip_code_prefix == "22790")
    .select("city")
    .distinct()
    .collect()
)
# Check if the value is unique
assert len(sellers_city_correction) == 1, f"geolocation_city contains {len(sellers_city_correction)} occurences for zip code prefix 22790"

# Get unique value
sellers_city_correction = sellers_city_correction[0]["city"]
sellers_city_correction

'rio de janeiro'

In [0]:
# Update sellers table:
df = df.withColumn(
    "seller_city",
    F.when(
        F.col("seller_city") == "04482255",
        sellers_city_correction
    ).otherwise(F.col("seller_city"))
)
# Assert that the change worked
assert (
    df
    .filter(df.seller_city == "04482255")
    .count() == 0
), "The correction did not work"

In [0]:
# Rename columns:
df = (
    df
    .withColumnRenamed("seller_zip_code_prefix", "zip_code_prefix")
    .withColumnRenamed("seller_city", "city")
    .withColumnRenamed("seller_state", "state")
)
df.limit(10).display()

seller_id,zip_code_prefix,city,state
3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
c240c4061717ac1806ae6ee72be3533b,20920,rio de janeiro,RJ
e49c26c3edfa46d227d5121a6b6e4d37,55325,brejao,PE
1b938a7ec6ac5061a66a3766e0e75f90,16304,penapolis,SP
768a86e36ad6aae3d03ee3c6433d61df,01529,sao paulo,SP
ccc4bbb5f32a6ab2b7066a4130f114e3,80310,curitiba,PR


In [0]:
# Save table in silver layer:
df.write.format("delta").mode("overwrite").saveAsTable(table_name)

Agora, para mostrar que realmente não temos mais esse erro:

In [0]:
%sql
SELECT * 
FROM mvp.silver.olist_sellers
WHERE city='04482255'

seller_id,zip_code_prefix,city,state


## Category Name Translation
Para esse trabalho, não iremos utilizar a tabela auxiliar de tradução do product_category_name. Sendo assim, iremos realizar uma verificação na tabela olist_product para ver se todos os valores da coluna product_category_name já estão em português. <br>
Caso a os valores já estiverem em portugês, nós não iremos passar a tabela olist_product_category_name_english para a camada silver.

In [0]:
%sql
--Check if all product_category_names are in portuguese
SELECT DISTINCT category_name 
FROM mvp.silver.olist_products
WHERE category_name IN (
    SELECT product_category_name_english
    FROM mvp.bronze.product_category_name_translation
)
AND category_name NOT IN (
    SELECT product_category_name
    FROM mvp.bronze.product_category_name_translation
)

category_name


Com base na busca acima, verificamos que todos os valores de product_category_name da tabela olist_product estão em portugês, sendo assim, não é necessário atualizar os valores da tabela de produtos.

## Validações finais:

### Verificando se todas as tabelas desejadas da camada bronze foram para a camada silver:

In [0]:
bronze_tables = get_tables_names_from_schema(table_schema="bronze")
silver_tables = get_tables_names_from_schema(table_schema="silver")

missing_tables = [t for t in silver_tables if t not in bronze_tables and t!="product_category_name_translation"]
assert len(missing_tables) == 0, f"Missing tables in silver layer: {missing_tables}"
print("All silver tables are valid")

All silver tables are valid


### Esquema das tabelas:
Por fim, de forma similar à como foi feito na camada bronze, foi utilizado a ia do databricks para fazer os comentários de cada coluna e das tabelas criadas na camada silver. 

In [0]:
for table_name in silver_tables:
    show_table_full_schema(
        table_name=table_name,
        table_schema="silver",
    )

SCHEMA FOR MVP.SILVER.OLIST_CUSTOMERS
+-------------+------------+---------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|table_catalog|table_schema|table_name     |comment                                                                                                                                                                                                                            |
+-------------+------------+---------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|mvp          |silver      |olist_customers|The table contains information about customers, including their identifiers and location details su

# Camada Ouro:
Para esse estudo, foi escolhido o modelo de flat_table, porquanto, no contexto das perguntas escolhidas, esse modelo simplifica o consumo e organização dos dados, reduzindo a necessidade de multiplos joins entre fatos e dimensões.<br>
Sendo assim, Para o contexto das perguntas a serem respondidas, serão feitas duas tabelas:
- sales_analysis:
    - order_id
    - order_item_id
    - product_id
    - category_name
    - photos_quantity
    - state
    - price

- order_review_analysis
    - order_id
    - customer_id
    - estimated_delivery_date
    - delivered_date
    - was_delayed
    - review_score


In [0]:
# Create catalog:
spark.sql("USE CATALOG mvp")
spark.sql("CREATE SCHEMA IF NOT EXISTS mvp.gold")
spark.sql ("USE SCHEMA gold")

DataFrame[]

## Sales_analysis

In [0]:
%sql
-- Sales_analysis table will be created with olist_products, olist_orders_items and olist_customers information, and the olist_orders will be used to link olist_custom to olist_order_items.
CREATE OR REPLACE TABLE sales_analysis AS
SELECT order_items.order_id,
    order_items.order_item_id,
    order_items.product_id,
    product.category_name,
    product.photos_quantity,
    customers.state,
    order_items.price
FROM mvp.silver.olist_order_items order_items
INNER JOIN mvp.silver.olist_products product ON order_items.product_id = product.product_id
INNER JOIN mvp.silver.olist_orders orders ON order_items.order_id = orders.order_id
INNER JOIN mvp.silver.olist_customers customers ON orders.customer_id = customers.customer_id


num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Check sales_analysis table
SELECT * 
FROM mvp.gold.sales_analysis
LIMIT 10


order_id,order_item_id,product_id,category_name,photos_quantity,state,price
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,cool_stuff,4,RJ,58.90
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,pet_shop,2,SP,239.90
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,moveis_decoracao,2,MG,199.00
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,perfumaria,1,SP,12.99
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,ferramentas_jardim,1,SP,199.90
00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,utilidades_domesticas,1,MG,21.90
00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,telefonia,1,SP,19.90
000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,ferramentas_jardim,3,SP,810.00
0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,beleza_saude,1,SP,145.95
0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,livros_tecnicos,1,SP,53.99


In [0]:
%sql
-- Check if there are a null value in price
SELECT count(price) 
FROM sales_analysis
WHERE price IS NULL


count(price)
0


## Order_Review_Analysis

In [0]:
%sql
-- order_review_analysis table will be created with mvp.silver.olist_orders e mvp.silver.olist_order_review, tendo uma coluna was_delay, que tera valor 1 para os casos aonde a data estimada foi menor que a data de entrega.
CREATE OR REPLACE TABLE order_review_analysis AS
SELECT orders.order_id,
    orders.customer_id,
    orders.estimated_delivery_date,
    orders.delivered_customer_date,
    CASE
        WHEN datediff(orders.delivered_customer_date, orders.estimated_delivery_date) > 0 THEN 1
        ELSE 0
    END AS was_delayed,
    review.score
FROM mvp.silver.olist_orders orders
INNER JOIN mvp.silver.olist_order_reviews review ON orders.order_id = review.order_id
WHERE (
    review.score IS NOT NULL
    AND orders.estimated_delivery_date IS NOT NULL
    AND orders.delivered_customer_date IS NOT NULL
    )


num_affected_rows,num_inserted_rows


In [0]:
%sql
--check order_review_analysis table
SELECT * 
FROM order_review_analysis
LIMIT 10

order_id,customer_id,estimated_delivery_date,delivered_customer_date,was_delayed,score
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,2017-10-18T00:00:00.000Z,2017-10-10T21:25:13.000Z,0,4
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,2018-08-13T00:00:00.000Z,2018-08-07T15:27:45.000Z,0,4
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,2018-09-04T00:00:00.000Z,2018-08-17T18:06:29.000Z,0,5
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,2017-12-15T00:00:00.000Z,2017-12-02T00:28:42.000Z,0,5
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,2018-02-26T00:00:00.000Z,2018-02-16T18:17:02.000Z,0,5
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,2017-08-01T00:00:00.000Z,2017-07-26T10:57:55.000Z,0,4
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,2017-06-07T00:00:00.000Z,2017-05-26T12:55:51.000Z,0,5
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,2017-03-06T00:00:00.000Z,2017-02-02T14:08:10.000Z,0,1
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,2017-08-23T00:00:00.000Z,2017-08-16T17:14:30.000Z,0,5
e6ce16cb79ec1d90b1da9085a6118aeb,494dded5b201313c64ed7f100595b95c,2017-06-07T00:00:00.000Z,2017-05-29T11:18:31.000Z,0,1


In [0]:
%sql
--check order_review_analysis table
SELECT * 
FROM order_review_analysis
WHERE was_delayed = 1
LIMIT 10

order_id,customer_id,estimated_delivery_date,delivered_customer_date,was_delayed,score
203096f03d82e0dffbc41ebc2e2bcfb7,d2b091571da224a1b36412c18bc3bbfe,2017-09-28T00:00:00.000Z,2017-10-09T22:23:46.000Z,1,2
fbf9ac61453ac646ce8ad9783d7d0af6,3a874b4d4c4b6543206ff5d89287f0c3,2018-03-12T00:00:00.000Z,2018-03-21T22:03:54.000Z,1,2
6ea2f835b4556291ffdc53fa0b3b95e8,c7340080e394356141681bd4c9b8fe31,2017-12-21T00:00:00.000Z,2017-12-28T18:59:23.000Z,1,1
66e4624ae69e7dc89bd50222b59f581f,684fa6da5134b9e4dab731e00011712d,2018-04-02T00:00:00.000Z,2018-04-03T13:28:46.000Z,1,1
a685d016c8a26f71a0bb67821070e398,911e4c37f5cafe1604fe6767034bf1ae,2017-03-30T00:00:00.000Z,2017-04-06T13:37:16.000Z,1,1
6a0a8bfbbe700284feb0845d95e0867f,68451b39b1314302c08c65a29f1140fc,2017-12-11T00:00:00.000Z,2017-12-28T19:43:00.000Z,1,1
a5474c0071dd5d1074e12d417078bbd0,ef15b3240b2083e0487762ee2978d2b8,2018-08-02T00:00:00.000Z,2018-08-03T19:28:47.000Z,1,5
9d531c565e28c3e0d756192f84d8731f,d4faa220408c20e53595d2950f361f3b,2017-12-22T00:00:00.000Z,2018-01-23T21:38:52.000Z,1,1
8fc207e94fa91a7649c5a5dab690272a,c69f8b33e62ecb30ff78ae46d7fb9241,2017-12-19T00:00:00.000Z,2018-01-20T13:42:22.000Z,1,3
33a3edb84b9df4cb49546859b990ac6d,35ec6c1ca9e5844c5ca94214cce16dca,2018-03-16T00:00:00.000Z,2018-03-22T00:03:53.000Z,1,1


In [0]:
%sql
--check order_review_analysis table
SELECT count(*) 
FROM order_review_analysis

count(*)
96359


In [0]:
%sql
--check order_review_analysis table
SELECT * 
FROM order_review_analysis
WHERE (
    estimated_delivery_date IS NULL
    OR delivered_customer_date IS NULL
    OR score IS NULL
)
LIMIT 10

order_id,customer_id,estimated_delivery_date,delivered_customer_date,was_delayed,score


## Esquema das tabelas:
De forma analoga a como foi feito nas camadas anteriores, foi usado a interface do Databricks para setar os comentarios das tabelas e das colunas faltantes (order_review_analysis.was_delayed), Isso foi feito utilizando a ia.<br>
O esquema das tabelas ficou da seguinte forma:

In [0]:
gold_table_names = get_tables_names_from_schema(table_schema="gold")
gold_table_names

['order_review_analysis', 'sales_analysis']

In [0]:
for table_name in gold_table_names:
    show_table_full_schema(
        table_name=table_name,
        table_schema="gold"
    )

SCHEMA FOR MVP.GOLD.ORDER_REVIEW_ANALYSIS
+-------------+------------+---------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|table_catalog|table_schema|table_name           |comment                                                                                                                                                                                                                                                                                                                                                          |
+-------------+------------+---------------------+--------------------------------------------------------------------------------------------------

# Resposta das Perguntas:
Na seção abaixo, serão feitas as queryes para respnder as perguntas citadas no inicio desse trabalho.

In [0]:
spark.sql("USE CATALOG mvp")
spark.sql ("USE SCHEMA gold")

DataFrame[]

## 1. Qual categoria gera mais faturamento?
Para essa pergunta, será feita uma query simples agrupando o category_name pela soma do preço, organizado de forma decrescente. Dessa forma serão mostradas as categorias que tiveram maior faturamento (soma dos preços).<br>
De acordo com a query abaixo, a categoria que gera maior faturamento é **beleza_saude**, que gera R$:1.258.681,34. Seguido por relogios_presentes com faturamento de R$:1.205.005,68 e  cama_mesa_banho com faturamento de R$:1.036.988,68


In [0]:
%sql
SELECT category_name, sum(price) AS revenue
from sales_analysis
GROUP BY category_name
ORDER BY revenue DESC

category_name,revenue
beleza_saude,1258681.34
relogios_presentes,1205005.68
cama_mesa_banho,1036988.68
esporte_lazer,988048.97
informatica_acessorios,911954.32
moveis_decoracao,729762.49
cool_stuff,635290.85
utilidades_domesticas,632248.66
automotivo,592720.11
ferramentas_jardim,485256.46


## 2. Produtos com mais fotos vendem mais?
Para responder essa pergunta, será feita um agrupamento da coluna photo_quantity da tabela sales_analysis, usando um contador para contar quantas vendas tiveram para cada quantidade de fotos.<br>
De acordo com a query abaixo, **produtos com mais fotos tendem a ter menos vendas**. os produtos com mais volume de venda possuem 1 foto, tendo um volume de venda de 56028, seguido por produtos com 2 fotos, com 21963 vendas, 3 fotos com 12392 vendas e 4 fotos com 8437 vendas.<br>
Além disso, é interessante notar e produtos com nenhuma foto tiveram 1603 vendas, ficando em 7 lugar, enquanto que produtos com mais de 10 fotos ficaram a partir de 12º lugar.

In [0]:
%sql
SELECT photos_quantity, count(*) AS seles_volume
FROM sales_analysis
GROUP BY photos_quantity
ORDER BY seles_volume DESC

photos_quantity,seles_volume
1,56028
2,21963
3,12392
4,8437
5,5368
6,3786
null,1603
7,1501
8,727
10,342


## 3. Produtos entregues com atrazo impactam na avaliação?
Para responder essa pergunta, será feito um agrupamento com o was_delayed, contendo o número de reviews, a média e o desvio padrão do score de revisão.<br>
De acordo com a query abaixo, **o atrazo na entrega de um produto possuem impacto na nota do review deixado**, sendo que produtos entregues em dia possuem uma nota média de 4.29 +- 1.15, enquanto que produtos entregues após o prazo estipulado tem uma média de 2.271 +- 1.571.

In [0]:
%sql

SELECT was_delayed,count(*) AS number_of_reviews, cast(mean(score) AS decimal(4, 3)) AS avg_score, cast(std(score) AS decimal(4, 3)) AS std_score
FROM order_review_analysis
GROUP BY was_delayed
ORDER BY was_delayed

was_delayed,number_of_reviews,avg_score,std_score
0,89949,4.290,1.150
1,6410,2.271,1.571


## 4. Quais estados possuem maior volume de venda? 
Para responder essa pergunta, será feito um agrupamento dos estados da tabela sales_analysis, contendo um contador para o volume de vendas, mostrado em ordem decrescente.<br>
De acordo com a query abaixo, **o estado com maior volume de vendas foi São Paulo, com  47449 vendas**, seguido pelo Rio de Janeiro, com 14579 vendas e Minas Gerais com 13129 vendas.

In [0]:
%sql

SELECT state, count(*) AS sales_volume
FROM sales_analysis
GROUP BY state
ORDER BY sales_volume DESC

state,sales_volume
SP,47449
RJ,14579
MG,13129
RS,6235
PR,5740
SC,4176
BA,3799
DF,2406
GO,2333
ES,2256


# Conclusão:
## Auto-Avaliação:
Esse trabalho conseguiu responder todas as perguntas formuladas no inicio de forma clara. Isso foi possivel devido às etapas de análise exploratória, que possibilitou a construção de um modelo de entidade-relacionamento, facilitando muito na etapa de construção do banco de dados e de tipagem durante a camada Silver.<br>

## Dificuldades:
Uma dificuldade encontrada durante esse trabalho foi na tabela de sales_review na camada bronze, porquanto a coluna com as mensagens das avaliações possuiam quebra de linha e outros caracteres, o que gerou um erro silencioso de deslocamento das colunas durante a leitura. Esse erro só foi encontrado durante a camada silver, quando foi realizado a tipagem das colunas e validação da perda de informação, porquanto tinham valores de review_id como timestamp. Para superar essa dificuldade, foi utilizado o notebook de análise exploratória, que mostrou que o erro estava na forma com que os dados estavam sendo lidos e não no dataset em sí. Com essa informação, foi realizado uma pesquisa no dataset junto com formas e opções de leitura e foi feito uma condicional, para ler somente essa tabela de forma diferente.<br>

## Futuros trabalhos:
Pensando nessa etapa, não foram excluidos as tabelas e informações não usadas na camada silver. Sendo assim, esse trabalho possue possibilidade de responder mas pergutas e gerar mais insights sobre esse dataset, por meio da criação de outras tabelas na camada gold. Além disso, para cada pergunta, podemos usar o pyspark junto com o pyplot/seaborn para gerar tabelas que melhorem a visualização desses dados, fazendo uma análise exploratória mais completa, e dos resultados, possibilitando uma melhor visualização das respostas.